In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
# Change this path if rerun.ipynb is not directly inside the Week6 folder
DATA_DIR = Path("./data_versions")

# Find all feature-version datasets
version_files = sorted(
    list(DATA_DIR.glob("v*.csv.gz")) +
    list(DATA_DIR.glob("v*.csv"))
)

print(f"Found {len(version_files)} dataset versions:")

for file_path in version_files:
    print(file_path.name)

Found 8 dataset versions:
v0_week5_baseline.csv.gz
v1_school_district.csv.gz
v2_hoa.csv.gz
v3_ratio.csv.gz
v4_school_hoa.csv.gz
v5_school_ratio.csv.gz
v6_all_features.csv.gz
version_summary.csv


In [3]:
def evaluate_linear_version(file_path):
    """
    Load one feature-set version, preprocess its features,
    train a Linear Regression model, and return evaluation metrics.
    """

    # Read one dataset version
    df = pd.read_csv(file_path)

    # Get a clean version name
    version_name = file_path.name

    if version_name.endswith(".csv.gz"):
        version_name = version_name[:-7]
    elif version_name.endswith(".csv"):
        version_name = version_name[:-4]

    # Verify the required columns
    required_columns = {"ClosePrice", "Dataset"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    # Replace positive and negative infinity with NaN
    df = df.replace([np.inf, -np.inf], np.nan)

    # Keep only rows assigned to Train or Test
    df = df[df["Dataset"].isin(["Train", "Test"])].copy()

    # Split using the existing Dataset column
    train_df = df[df["Dataset"] == "Train"].copy()
    test_df = df[df["Dataset"] == "Test"].copy()

    if train_df.empty:
        raise ValueError("The training dataset is empty.")

    if test_df.empty:
        raise ValueError("The test dataset is empty.")

    # Remove rows with missing or invalid target values
    train_df = train_df[
        train_df["ClosePrice"].notna() &
        np.isfinite(train_df["ClosePrice"]) &
        (train_df["ClosePrice"] > 0)
    ].copy()

    test_df = test_df[
        test_df["ClosePrice"].notna() &
        np.isfinite(test_df["ClosePrice"]) &
        (test_df["ClosePrice"] > 0)
    ].copy()

    if train_df.empty or test_df.empty:
        raise ValueError(
            "No valid target rows remain after filtering ClosePrice."
        )

    # Create target arrays
    y_train = train_df["ClosePrice"].to_numpy(dtype=np.float64)
    y_test = test_df["ClosePrice"].to_numpy(dtype=np.float64)

    # Columns that should not be used as model inputs
    columns_to_drop = [
        "ClosePrice",
        "Dataset",
        "CloseDate"
    ]

    X_train = train_df.drop(
        columns=columns_to_drop,
        errors="ignore"
    ).copy()

    X_test = test_df.drop(
        columns=columns_to_drop,
        errors="ignore"
    ).copy()

    # Make sure test columns are aligned with training columns
    X_test = X_test.reindex(columns=X_train.columns)

    # Drop columns that are completely missing in the training data
    all_missing_columns = X_train.columns[
        X_train.isna().all()
    ].tolist()

    if all_missing_columns:
        X_train = X_train.drop(columns=all_missing_columns)
        X_test = X_test.drop(
            columns=all_missing_columns,
            errors="ignore"
        )

    if X_train.shape[1] == 0:
        raise ValueError(
            "No usable predictor columns remain."
        )

    # Store the raw feature count
    feature_columns = X_train.columns.tolist()

    # Automatically identify numeric and categorical features
    numeric_columns = X_train.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    categorical_columns = X_train.select_dtypes(
        exclude=["number", "bool"]
    ).columns.tolist()

    # Convert boolean columns to integers
    for column in numeric_columns:
        if X_train[column].dtype == bool:
            X_train[column] = X_train[column].astype(int)
            X_test[column] = X_test[column].astype(int)

    # Numeric preprocessing
    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median")
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]
    )

    # Categorical preprocessing
    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent")
            ),
            (
                "one_hot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    drop="first"
                )
            )
        ]
    )

    # Only add transformers that actually have columns
    transformers = []

    if numeric_columns:
        transformers.append(
            (
                "numeric",
                numeric_pipeline,
                numeric_columns
            )
        )

    if categorical_columns:
        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns
            )
        )

    if not transformers:
        raise ValueError(
            "No numeric or categorical predictors were found."
        )

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    # Build the complete model pipeline
    model_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "model",
                LinearRegression()
            )
        ]
    )

    # Fit only on the training data
    model_pipeline.fit(X_train, y_train)

    # Generate predictions
    train_pred = model_pipeline.predict(X_train)
    test_pred = model_pipeline.predict(X_test)

    # Calculate absolute percentage errors
    train_ape = np.abs(
        (y_train - train_pred) / y_train
    )

    test_ape = np.abs(
        (y_test - test_pred) / y_test
    )

    # Count the number of columns after preprocessing
    encoded_feature_count = (
        model_pipeline
        .named_steps["preprocessor"]
        .transform(X_train.iloc[:1])
        .shape[1]
    )

    # Return one row of results
    return {
        "Version": version_name,
        "Model": "Linear Regression",
        "Rows": len(df),
        "Train Rows": len(train_df),
        "Test Rows": len(test_df),
        "Features Before Encoding": len(feature_columns),
        "Numeric Features": len(numeric_columns),
        "Categorical Features": len(categorical_columns),
        "Encoded Features": encoded_feature_count,
        "Train R2": r2_score(y_train, train_pred),
        "Test R2": r2_score(y_test, test_pred),
        "Train RMSE": mean_squared_error(
            y_train,
            train_pred
        ) ** 0.5,
        "Test RMSE": mean_squared_error(
            y_test,
            test_pred
        ) ** 0.5,
        "Train MAPE": mean_absolute_percentage_error(
            y_train,
            train_pred
        ),
        "Test MAPE": mean_absolute_percentage_error(
            y_test,
            test_pred
        ),
        "Train MdAPE": np.median(train_ape),
        "Test MdAPE": np.median(test_ape)
    }

In [4]:
linear_results = []
failed_versions = []

for file_path in version_files:
    print("=" * 80)
    print(f"Running Linear Regression: {file_path.name}")

    try:
        result = evaluate_linear_version(file_path)

        linear_results.append(result)

        print(
            "Completed | "
            f"Test R2: {result['Test R2']:.4f} | "
            f"Test RMSE: ${result['Test RMSE']:,.0f} | "
            f"Test MAPE: {result['Test MAPE']:.2%} | "
            f"Test MdAPE: {result['Test MdAPE']:.2%}"
        )

    except Exception as error:
        failed_versions.append(
            {
                "Version": file_path.name,
                "Error": str(error)
            }
        )

        print(f"Failed: {file_path.name}")
        print(f"Reason: {error}")

Running Linear Regression: v0_week5_baseline.csv.gz


Completed | Test R2: 0.7088 | Test RMSE: $829,271 | Test MAPE: 38.38% | Test MdAPE: 21.81%
Running Linear Regression: v1_school_district.csv.gz


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Completed | Test R2: 0.7389 | Test RMSE: $785,197 | Test MAPE: 33.44% | Test MdAPE: 18.60%
Running Linear Regression: v2_hoa.csv.gz


Completed | Test R2: 0.7106 | Test RMSE: $826,613 | Test MAPE: 38.02% | Test MdAPE: 21.66%
Running Linear Regression: v3_ratio.csv.gz


...

Completed | Test R2: 0.7402 | Test RMSE: $783,316 | Test MAPE: 33.06% | Test MdAPE: 18.49%
Running Linear Regression: v5_school_ratio.csv.gz


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Completed | Test R2: 0.7400 | Test RMSE: $783,611 | Test MAPE: 33.36% | Test MdAPE: 18.30%
Running Linear Regression: v6_all_features.csv.gz


/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [0, 1, 2] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Completed | Test R2: 0.7412 | Test RMSE: $781,817 | Test MAPE: 32.96% | Test MdAPE: 18.17%
Running Linear Regression: version_summary.csv
Failed: version_summary.csv
Reason: Missing required columns: ['ClosePrice', 'Dataset']


In [5]:
if linear_results:
    linear_results_df = pd.DataFrame(linear_results)

    linear_results_df = linear_results_df.sort_values(
        by="Test R2",
        ascending=False
    ).reset_index(drop=True)

    display(
        linear_results_df[
            [
                "Version",
                "Features Before Encoding",
                "Encoded Features",
                "Train R2",
                "Test R2",
                "Test RMSE",
                "Test MAPE",
                "Test MdAPE"
            ]
        ]
    )
else:
    print("No dataset version completed successfully.")

,Version,Features Before Encoding,Encoded Features,Train R2,Test R2,Test RMSE,Test MAPE,Test MdAPE
0,v6_all_features,993,1670,0.773936,0.741151,781817.135775,0.329551,0.181718
1,v4_school_hoa,992,1669,0.772690,0.740157,783316.012229,0.330634,0.184912
2,v5_school_ratio,992,1669,0.771966,0.739961,783611.290541,0.333567,0.182984
3,v1_school_district,991,1668,0.770752,0.738908,785196.993288,0.334419,0.186042
4,v2_hoa,989,989,0.729531,0.710638,826612.658679,0.380209,0.216597
5,v3_ratio,989,989,0.728657,0.710005,827516.969860,0.382715,0.213848
6,v0_week5_baseline,988,988,0.727569,0.708774,829271.023178,0.383832,0.218131


In [6]:
linear_results_df[
    [
        "Version",
        "Features Before Encoding",
        "Encoded Features",
        "Train R2",
        "Test R2",
        "Test RMSE",
        "Test MAPE",
        "Test MdAPE"
    ]
]

,Version,Features Before Encoding,Encoded Features,Train R2,Test R2,Test RMSE,Test MAPE,Test MdAPE
0,v6_all_features,993,1670,0.773936,0.741151,781817.135775,0.329551,0.181718
1,v4_school_hoa,992,1669,0.772690,0.740157,783316.012229,0.330634,0.184912
2,v5_school_ratio,992,1669,0.771966,0.739961,783611.290541,0.333567,0.182984
3,v1_school_district,991,1668,0.770752,0.738908,785196.993288,0.334419,0.186042
4,v2_hoa,989,989,0.729531,0.710638,826612.658679,0.380209,0.216597
5,v3_ratio,989,989,0.728657,0.710005,827516.969860,0.382715,0.213848
6,v0_week5_baseline,988,988,0.727569,0.708774,829271.023178,0.383832,0.218131
